# RAY-IMAGE v0.1 — Google Colab

This notebook connects the GitHub repository to a Colab runtime, verifies CUDA, runs the architecture smoke test, creates the synthetic dataset, and launches a small training run.

**Important:** Select **Runtime → Change runtime type → GPU** before running the cells. If CUDA is unavailable, the notebook stops before wasting time on CPU training.

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('No CUDA GPU is attached. In Colab choose Runtime → Change runtime type → GPU, reconnect, then run again.')
print('GPU:', torch.cuda.get_device_name(0))
print('VRAM GiB:', round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 2))

In [ ]:
%cd /content
!rm -rf anime-ai-companion
!git clone https://github.com/Rishidev-20thcenturey/anime-ai-companion.git
%cd /content/anime-ai-companion
!pip install -q -r requirements.txt

In [ ]:
!python -m ray_image.train_smoke

In [ ]:
!python tools/make_toy_dataset.py --output data/toy --samples 64 --size 64

In [ ]:
!python -m ray_image.train --manifest data/toy/manifest.jsonl --steps 200 --batch-size 8 --save /content/ray_image_v0_1.pt

In [ ]:
from pathlib import Path
checkpoint = Path('/content/ray_image_v0_1.pt')
print('checkpoint:', checkpoint.exists())
if checkpoint.exists():
    print('size MiB:', round(checkpoint.stat().st_size / 1024**2, 2))

## Next

This 200-step run is only an infrastructure test. Once it works on GPU, we will replace the synthetic shapes with a real captioned image dataset, split VAE pretraining from DiT training, and add validation image generation.